# **03 - Evaluación de Modelos y Visualización de Resultados Finales**
## **Proyecto: Impacto de la Inteligencia Artificial Generativa en Estudiantes**

Este notebook aborda la fase final de visualización, comparación de métricas y cierre del proyecto:
1. **Carga del dataset procesado y los modelos serializados**.
2. **Evaluación de predicciones** en el conjunto de prueba independiente ($Test$).
3. **Generación de reportes** de métricas ($R^2$ y RMSE).
4. **Visualizaciones de rendimiento** comparativas.
5. **Conclusiones del análisis**.

In [ ]:
# ============================================================
# CONFIGURACIÓN GENERAL E IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import os
import sys
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Agregar directorio raíz para importar el paquete src
sys.path.append(os.path.abspath('..'))

import src.plots as plots

print('Entorno de trabajo inicializado correctamente.')

### 1. Carga de Datos y Modelos Guardados

In [ ]:
# Reconstruimos X_test e y_test de forma idéntica usando la semilla 42
df = pd.read_csv('../data/processed/ai_student_impact_procesado.csv')
X = df.drop(columns=['Post_Semester_GPA'])
y = df['Post_Semester_GPA']

_, X_test, _, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Cargamos los modelos desde models/
lm_base = joblib.load('../models/baseline_model.pkl')
pipeline_multi = joblib.load('../models/multiple_linear_model.pkl')
pipeline_poly = joblib.load('../models/polynomial_linear_model.pkl')
pipeline_ridge = joblib.load('../models/ridge_model.pkl')
pipeline_lasso = joblib.load('../models/lasso_model.pkl')
best_model = joblib.load('../models/optimized_best_model.pkl')

print('Conjunto de prueba y modelos cargados con éxito.')

### 2. Generación de Predicciones y Evaluación Comparativa en Test

Evaluamos el desempeño de todos los modelos entrenados y generamos un reporte en `outputs/reports/`.

In [ ]:
modelos_nombres = [
    'Baseline (Simple)',
    'Regresión Lineal Múltiple',
    'Regresión Polinomial (Grado 2)',
    'Regularización Ridge (α=10.0)',
    'Regularización Lasso (α=0.01)',
    'Ridge G2 Optimizado (GridSearchCV)'
]

modelos_instancias = [
    None,  # Manual
    pipeline_multi,
    pipeline_poly,
    pipeline_ridge,
    pipeline_lasso,
    best_model
]

r2_test_scores = []
rmse_test_scores = []

for name, model in zip(modelos_nombres, modelos_instancias):
    if name == 'Baseline (Simple)':
        pred = lm_base.predict(X_test[['Pre_Semester_GPA']])
    else:
        pred = model.predict(X_test)
    r2_test_scores.append(r2_score(y_test, pred))
    rmse_test_scores.append(np.sqrt(mean_squared_error(y_test, pred)))

df_comparacion = pd.DataFrame({
    'Modelo': modelos_nombres,
    'R² Test': r2_test_scores,
    'RMSE Test': rmse_test_scores
}).sort_values(by='R² Test', ascending=False)

# Guardar reporte
df_comparacion.to_csv('../outputs/reports/metrics_comparison.csv', index=False)
print('Comparación de rendimiento:')
display(df_comparacion.round(5))

### 3. Visualizaciones Finales

Dibujamos la comparativa visual y las predicciones finales del modelo ganador.

In [ ]:
# Gráfico de comparación de R²
plots.plot_model_comparison(df_comparacion, filepath='../outputs/figures/model_comparison.png')

# Gráfico real vs predicho para el mejor modelo
best_pred = best_model.predict(X_test)
plots.plot_predictions_vs_real(y_test, best_pred, 
                               'Modelo Ganador (Ridge Optimizado): Reales vs Predichos en Test',
                               filepath='../outputs/figures/final_model_predictions_vs_real.png')

### 4. Conclusiones Principales

1. **Impacto de las Variables**: Las variables que muestran mayor capacidad predictiva sobre el GPA posterior (`Post_Semester_GPA`) son el GPA inicial (`Pre_Semester_GPA`), las horas de estudio tradicional (`Traditional_Study_Hours`), y las variables asociadas a la IA generativa (`Weekly_GenAI_Hours` y `Tool_Diversity`). Esto demuestra una fuerte interacción entre los hábitos tradicionales y las nuevas competencias tecnológicas.
2. **Ajuste del Modelo**: El paso de un modelo Baseline simple ($R^2 \approx 0.35$ en test) a un modelo de Regresión Lineal Múltiple incrementó drásticamente el rendimiento ($R^2 \approx 0.88$). Al incluir interacciones polinomiales de segundo grado, logramos capturar efectos no lineales en la interacción de horas de estudio tradicionales y de IA, elevando el rendimiento a un $R^2 \approx 0.94$.
3. **Regularización y Optimización**: La validación cruzada y la regularización Ridge/Lasso confirmaron que el modelo polinomial de grado 2 es sumamente estable. La sintonización mediante `GridSearchCV` identificó que un polinomio de grado 2 con regularización Ridge ligera ($alpha = 10.0$) entrega la mejor capacidad de generalización en el conjunto de prueba independiente ($R^2 \approx 0.94$), controlando cualquier riesgo de sobreajuste.